In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

train_woe = pd.read_csv("../data/processed/train_woe.csv")
validation_woe = pd.read_csv("../data/processed/validation_woe.csv")
test_woe = pd.read_csv("../data/processed/test_woe.csv")

target = "SeriousDlqin2yrs"

final_variables = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "MonthlyIncome",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents",
    "NumberOfTimes90DaysLate"
]

In [ ]:
X_train = sm.add_constant(train_woe[final_variables])
y_train = train_woe[target]

model = sm.Logit(y_train, X_train).fit()

model.summary()

In [ ]:
PDO = 20
BASE_SCORE = 600
BASE_ODDS = 20

factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)

def add_pd_and_score(df, model, variables):
    X = sm.add_constant(df[variables])
    
    df = df.copy()
    df["pd"] = model.predict(X)
    df["logit"] = np.dot(X, model.params)
    df["score"] = offset - factor * df["logit"]
    
    return df

In [ ]:
train_scored = add_pd_and_score(train_woe, model, final_variables)
validation_scored = add_pd_and_score(validation_woe, model, final_variables)
test_scored = add_pd_and_score(test_woe, model, final_variables)

In [ ]:
score_distribution = pd.DataFrame({
    "dataset": ["Train", "Validation", "Test"],
    "count": [
        len(train_scored),
        len(validation_scored),
        len(test_scored)
    ],
    "mean_score": [
        train_scored["score"].mean(),
        validation_scored["score"].mean(),
        test_scored["score"].mean()
    ],
    "std_score": [
        train_scored["score"].std(),
        validation_scored["score"].std(),
        test_scored["score"].std()
    ],
    "min_score": [
        train_scored["score"].min(),
        validation_scored["score"].min(),
        test_scored["score"].min()
    ],
    "p25_score": [
        train_scored["score"].quantile(0.25),
        validation_scored["score"].quantile(0.25),
        test_scored["score"].quantile(0.25)
    ],
    "median_score": [
        train_scored["score"].median(),
        validation_scored["score"].median(),
        test_scored["score"].median()
    ],
    "p75_score": [
        train_scored["score"].quantile(0.75),
        validation_scored["score"].quantile(0.75),
        test_scored["score"].quantile(0.75)
    ],
    "max_score": [
        train_scored["score"].max(),
        validation_scored["score"].max(),
        test_scored["score"].max()
    ],
    "bad_rate": [
        train_scored[target].mean(),
        validation_scored[target].mean(),
        test_scored[target].mean()
    ]
})

In [ ]:
score_distribution

## Decile Analysis за Train / Validation / Test

In [ ]:
def decile_analysis(df, dataset_name):
    df = df.copy()
    
    df["score_decile"] = pd.qcut(
        df["score"],
        q=10,
        labels=False,
        duplicates="drop"
    )
    
    result = (
        df
        .groupby("score_decile")
        .agg(
            customers=("score", "count"),
            min_score=("score", "min"),
            max_score=("score", "max"),
            avg_score=("score", "mean"),
            avg_pd=("pd", "mean"),
            bad_rate=(target, "mean")
        )
        .reset_index()
    )
    
    result["dataset"] = dataset_name
    
    return result

In [ ]:
train_deciles = decile_analysis(train_scored, "Train")
validation_deciles = decile_analysis(validation_scored, "Validation")
test_deciles = decile_analysis(test_scored, "Test")

all_deciles = pd.concat(
    [train_deciles, validation_deciles, test_deciles],
    ignore_index=True
)

all_deciles

In [ ]:
validation_deciles

In [ ]:
plt.figure(figsize=(10, 6))

for dataset_name, decile_df in [
    ("Train", train_deciles),
    ("Validation", validation_deciles),
    ("Test", test_deciles)
]:
    plt.plot(
        decile_df["score_decile"],
        decile_df["bad_rate"],
        marker="o",
        label=dataset_name
    )

plt.xlabel("Score Decile")
plt.ylabel("Observed Bad Rate")
plt.title("Observed Bad Rate by Score Decile")
plt.legend()
plt.show()

## Cut-off fit

In [ ]:
def cutoff_summary(df, cutoff_score, dataset_name):
    approved = df[df["score"] >= cutoff_score]
    rejected = df[df["score"] < cutoff_score]
    
    return {
        "dataset": dataset_name,
        "cutoff_score": cutoff_score,
        "approval_rate": len(approved) / len(df),
        "rejection_rate": len(rejected) / len(df),
        "approved_bad_rate": approved[target].mean(),
        "rejected_bad_rate": rejected[target].mean(),
        "bad_capture_rate": rejected[target].sum() / df[target].sum()
    }

In [ ]:
cutoff_score = 600

cutoff_fit = pd.DataFrame([
    cutoff_summary(train_scored, cutoff_score, "Train"),
    cutoff_summary(validation_scored, cutoff_score, "Validation"),
    cutoff_summary(test_scored, cutoff_score, "Test")
])

cutoff_fit

In [ ]:
train_scored.to_csv("../data/processed/train_scored.csv", index=False)
validation_scored.to_csv("../data/processed/validation_scored.csv", index=False)
test_scored.to_csv("../data/processed/test_scored.csv", index=False)

score_distribution.to_csv("../data/outputs/score_distribution.csv", index=False)
all_deciles.to_csv("../data/outputs/score_decile_analysis.csv", index=False)
cutoff_fit.to_csv("../data/outputs/cutoff_fit_summary.csv", index=False)

## Score Run & Fit Summary

The final scorecard was applied to the train, validation and test samples.

Score distributions are highly stable across all datasets. Mean score values are almost identical across train, validation and test, indicating no material sample shift.

The decile analysis confirms strong rank ordering. The lowest score decile shows an observed bad rate of approximately 31%, while the highest score decile shows an observed bad rate below 1%.

At the proposed cut-off score of 600, the scorecard produces stable credit policy outcomes across all samples:

- Approval Rate: approximately 65.5%
- Approved Portfolio Bad Rate: approximately 1.9% - 2.0%
- Bad Capture Rate: approximately 81%

The results indicate that the selected cut-off is stable and that the scorecard generalizes well across unseen samples.